In [ ]:
import pandas as pd

# Path to data frame with WSIs
df_path = "D:\DATA\with_snomed_category.csv"
df_all = pd.read_csv(df_path)
print(df_all.columns)

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

In [ ]:
from helper_functions import subset_df, subset_df_list
df_HE = df_all.copy()
#df_HE = subset_df(df_all, "stain", "HE")
#df_HE = subset_df(df_all, "mattype tekst", "Hist. store")
#df_HE = subset_df_list(df_HE, "T_category", "Placenta, Fetal Membranes, and Fetus")

In [ ]:
from helper_functions import with_tissue_artifact

# Path to cache file
cache_file = "cache_all_slides.pkl"

df_qc = with_tissue_artifact(df_HE, cache_file, segmentation_type = "artifact", status="complete", version="default")
with_qc = list(set(df_qc["filename"].tolist()))
print("WSIs with completed artifact detection: ", len(with_qc))

df_sub = df_HE[df_HE["filename"].isin(with_qc)].copy()

In [ ]:
from helper_functions import load_big_cache

big_cache = load_big_cache(cache_file)

def artifact_percentages_from_cache(big_cache, version="default"):
    rows = []
    for slide_name, categories in big_cache.items():
        if "artifact" not in categories:
            continue
        if version not in categories["artifact"]:
            continue

        entry = categories["artifact"][version]
        if entry.get("status") != "complete":
            continue

        df = entry["df"]

        for artifact_class, row in df.iterrows():
            rows.append({
                "slide": slide_name,
                "artifact_class": artifact_class,
                "percentage": row["percentage"]
            })

    return pd.DataFrame(rows)

In [ ]:
artifact_df = artifact_percentages_from_cache(big_cache, version="default")
print(artifact_df.head(10))

In [ ]:
# Only include slides from artifact_df which are in df_sub
slides = artifact_df["slide"].astype(str)

# true if filename appears at least once in df_sub
mask = slides.apply(lambda s: df_sub["filename"].str.endswith(s).any())
artifact_sub = artifact_df[mask]
unique = artifact_sub["slide"].nunique()

print("artifact_sub: ", len(artifact_sub), 
      "unique files i artifact_sub: ", unique, 
      "files in df_sub: ", len(df_sub))

In [ ]:
artifact_sub.groupby("artifact_class")["percentage"].describe()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
sns.boxplot(
    data=artifact_sub,
    x="artifact_class",
    y="percentage",
    palette="Pastel1"
)
plt.ylabel("Percentage of tissue area (%)")
plt.xlabel("Artifact type")
plt.title("Distribution of artifact area across slides")
plt.xticks(rotation=45, ha="right")
plt.ylim(0,100)
plt.tight_layout()
plt.show()

In [ ]:
pivot_df = artifact_sub.pivot(
    index="slide",
    columns="artifact_class",
    values="percentage"
)

In [ ]:
identify_slides = (
    pivot_df[(pivot_df["Edge & Air Bubble"] > 20) & (pivot_df["Out of Focus"] < 20)]
    .sort_values("Out of Focus")
    .tail(5)
)
print(identify_slides)

In [ ]:
slides = [
    "slide1.mrxs", # Dark > 40%, OOF < 50%
    "slide2.mrxs", # Edge >20%, OOF < 20%
]

In [ ]:
matched = df_sub[df_sub["filename"].str.endswith(tuple(slides))]["filename"].tolist()

print(matched)

In [ ]:
artifact_df[artifact_df["slide"].isin(slides)]

In [ ]:
# Visualize Single Slide Classification
import os
from wsidata import open_wsi

path = matched[0]
print(path)

zarr_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"
zarr_path = os.path.join(zarr_dir, os.path.basename(path).replace(".mrxs", ".zarr"))
wsi = open_wsi(path, zarr_path)
wsi

In [ ]:
import lazyslide as zs 

TISSUE_KEY = 'tissue_default'
zs.pl.tissue(wsi, tissue_key = TISSUE_KEY, show_contours=True)

In [ ]:
ARTIFACT_KEY = "artifacts_grandqc"

viewer = zs.pl.WSIViewer(wsi)
viewer.add_image('thumbnail')
viewer.add_polygons(ARTIFACT_KEY, color_by='class', alpha=0.4)
viewer.show()

In [ ]:
ARTIFACT_KEY = "artifacts_grandqc"

viewer = zs.pl.WSIViewer(wsi)
viewer.add_image('thumbnail')
#viewer.add_polygons(ARTIFACT_KEY, color_by='class', alpha=0.5)
#viewer.add_zoom(tissue_id=0, tissue_key=TISSUE_KEY)
viewer.add_zoom(xmin=89000, xmax=89500, ymin=250000, ymax=250500, tissue_key=TISSUE_KEY)
viewer.show()